# TP3 (a completer) : Regression lineaire — *California Housing*

Remplacez chaque `...` et chaque `# TODO`. Corrige :
`../notebooks/TP3_regression_lineaire.ipynb`.

**Objectif.** Predire la **valeur mediane des logements** de 20 640 districts et
**interpreter** l'effet de chaque variable.

In [ ]:
# Cellule fournie : a executer telle quelle.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NAVY, ACCENT, GRAY = "#16284D", "#0EA5E9", "#5B6679"
RED = "#C0504D"
PALETTE = [ACCENT, NAVY, "#F79646", "#3FA45B", RED]
plt.rcParams.update({
    "figure.figsize": (7, 4.5), "font.size": 12,
    "axes.titlecolor": NAVY, "axes.titleweight": "bold",
    "axes.edgecolor": GRAY, "axes.spines.top": False, "axes.spines.right": False,
})
pd.set_option("display.width", 120)
print("Environnement pret.")

## Etape 0 : charger les donnees (fournie)

In [ ]:
from sklearn.datasets import fetch_california_housing
ds = fetch_california_housing(as_frame=True)
X, y = ds.data, ds.target.rename("prix")   # prix en x100 000 USD
X.head()

## 1. Exploration
**Consigne.** Affichez la correlation de chaque variable avec `prix`, triee.

In [ ]:
# TODO : correlations avec le prix (indice : pd.concat([X, y], axis=1).corr())
pd.concat([X, y], axis=1).corr()["prix"].sort_values(ascending=False)

## 2. Modelisation
**Consigne.** Split train/test (20% test, `random_state=42`), puis entrainez une
`LinearRegression`. Affichez l'ordonnee a l'origine et les coefficients.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
modele = LinearRegression().fit(X_train, y_train)       # TODO : entrainer la regression
# TODO : afficher intercept_ et coef_
print("Ordonnee a l'origine :", modele.intercept_)
print(pd.Series(modele.coef_, index=X.columns))

## 3. Evaluation
**Consigne.** Calculez **R2**, **RMSE** et **MAE** sur le test.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = modele.predict(X_test)
# TODO : R2, RMSE (= racine de mean_squared_error), MAE
print("R2   :", r2_score(y_test, y_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE  :", mean_absolute_error(y_test, y_pred))

## 4. Visualisation
**Consigne.** (a) Tracez `prix` vs `MedInc` avec la droite de regression simple.
(b) Tracez **predit vs reel** sur un echantillon. (c) Tracez les **residus**.

In [ ]:
# (a) prix vs MedInc + droite
simple = LinearRegression().fit(X[["MedInc"]], y)
xs = np.linspace(0, X["MedInc"].quantile(0.99), 100).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(6.2, 4.6))
ax.scatter(X["MedInc"], y, s=8, color=ACCENT, alpha=0.2, edgecolor="none")
# TODO : tracer la droite simple.predict(xs)
ax.plot(xs, simple.predict(xs), color=RED, lw=2)
ax.set(title="Prix vs revenu median", xlabel="MedInc", ylabel="prix (x100k USD)")
plt.show()

In [ ]:
# (b) predit vs reel (echantillon de 2000 points)
rng = np.random.default_rng(0)
idx = rng.choice(len(y_test), 2000, replace=False)
yt, yp = y_test.to_numpy()[idx], y_pred[idx]
fig, ax = plt.subplots(figsize=(5.4, 5))
# TODO : scatter(yt, yp) + diagonale de reference
ax.scatter(yt, yp, s=8, color=ACCENT, alpha=0.3, edgecolor="none")
ax.plot([yt.min(), yt.max()], [yt.min(), yt.max()], color=RED, lw=2)
ax.set(title="Predit vs reel", xlabel="prix reel", ylabel="prix predit")
plt.show()

In [ ]:
# (c) residus
residus = yt - yp
fig, ax = plt.subplots(figsize=(6.2, 4.4))
ax.axhline(0, color=NAVY, lw=2)
# TODO : scatter(yp, residus)
ax.scatter(yp, residus, s=8, color=ACCENT, alpha=0.3, edgecolor="none")
ax.set(title="Residus", xlabel="prix predit", ylabel="residu")
plt.show()

## 5. Interpretation
**Consigne.** Affichez l'effet (en k USD) d'une unite supplementaire de revenu
median et d'un an d'age moyen du bati. Estimez le prix de 3 districts.

In [ ]:
coefs = pd.Series(modele.coef_, index=X.columns)
# TODO : afficher coefs['MedInc']*100 et coefs['HouseAge']*100
print("Effet +1 de revenu median :", coefs["MedInc"] * 100, "k USD")
print("Effet +1 an d'age du bati :", coefs["HouseAge"] * 100, "k USD")


estim = X.head(3).copy()
estim["prix_estime"] = ...    # TODO : modele.predict(X.head(3))
estim

## A rendre
- R2, RMSE, MAE et leur interpretation.

R² = 0.58, RMSE = 0.75 (~75k USD), MAE = 0.53 (~53k USD). Le modèle explique un peu plus de la moitié des variations de prix ; en moyenne il se trompe d'environ 53k USD par district. Honnête pour un modèle linéaire simple, mais loin d'être parfait.

- Ce que revele le graphique des residus (indice : effet de plafond).

on observe une bande diagonale nette en haut → c'est l'effet de plafond. Les prix du dataset sont plafonnés à 5 (500k USD) : le modèle prédit des valeurs au-delà ou en deçà pour ces districts saturés, ce qui crée ces résidus alignés. Le modèle ne peut pas bien gérer cette troncature.

- L'effet du revenu median sur le prix.

 c'est la variable la plus influente. Chaque unité supplémentaire de MedInc augmente le prix prédit d'environ +40k USD, toutes choses égales par ailleurs.

**Bonus.** Comparez le R2 avec une seule variable (`MedInc`) vs toutes.